In [ ]:
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score

In [ ]:
!file final.parquet

final.parquet: Apache Parquet


In [ ]:
!ls -la

total 335048
drwxr-xr-x 1 root root      4096 Apr  6 04:38 .
drwxr-xr-x 1 root root      4096 Apr  6 03:19 ..
drwxr-xr-x 4 root root      4096 Apr  2 13:31 .config
-rw-r--r-- 1 root root 206252870 Apr  6 04:39 final_long.parquet
-rw-r--r-- 1 root root 136810917 Apr  6 04:39 final.parquet
drwxr-xr-x 1 root root      4096 Apr  2 13:31 sample_data


In [ ]:
df = pd.read_parquet('final.parquet')

In [ ]:
df.head()

,question,source,gemini_answers,gpt_answers,claude_answers,human_answers,kimi_answers,gemini_to_claude,gpt_to_claude,claude_to_claude,...,human_to_gemini,claude_to_gemini,gemini_to_gemini,gpt_to_gemini,kimi_to_gemini,human_to_kimi,claude_to_kimi,gemini_to_kimi,gpt_to_kimi,kimi_to_kimi
0,what composer used sound mass,open_qa,The most prominent composer strongly associate...,"Several 20th-century composers used the ""sound...",# Composers Who Used Sound Mass\n\n**Gyorgy Li...,"Composers and works include Barbara Kolb , Pau...","The term **""sound mass""** (or **""sound-mass co...",The most prominent composer associated with th...,"Several 20th-century composers pioneered the ""...",# Composers Who Used Sound Mass\n\n**György Li...,...,Early examples of sound mass techniques includ...,# Composers and the Sound Mass Technique\n\n**...,**György Ligeti** is the most prominent compos...,"Several 20th-century composers pioneered the ""...","The term **""sound mass""** (or **""sound-mass co...",Representative practitioners range from Barbar...,György Ligeti dominates any discussion of the ...,György Ligeti stands as the composer most famo...,Twentieth-century composers reoriented musical...,“Sound mass”—also “sound-mass composition”—is ...
1,where did the persian war take place,open_qa,The Persian Wars primarily took place in **mai...,If you mean the Greco–Persian Wars (often call...,# The Persian Wars\n\nThe Persian Wars took pl...,The Greco-Persian Wars (also often called the ...,The **Persian Wars** (also called the **Greco-...,The Persian Wars primarily took place across *...,If you're referring to the Greco-Persian Wars ...,# The Persian Wars\n\nThe Persian Wars (5th ce...,...,"The Greco-Persian Wars, also known as the Pers...","# The Persian Wars\n\nThe Persian Wars, a seri...",The Persian Wars primarily unfolded across mai...,If you're referring to the Greco-Persian Wars ...,"The **Greco-Persian Wars**, also known as the ...","From 499 to 449 BC, the Achaemenid Empire clas...","# Persian Wars, 492–479 BCE\n\nBetween 492 and...",The Persian Wars unfolded across three intertw...,Greco-Persian Wars (499–449 BCE) were fought c...,"Greco-Persian Wars, 499–449 BCE \nConflict zo..."
2,what are add ons,open_qa,An **add-on** is something extra or additional...,"""Add-ons"" (sometimes written ""add ons"", ""addon...",# Add-ons\n\nAdd-ons are **additional features...,"Plug-in (computing) , a piece of software whic...",**Add-ons** are extra software components or e...,An **add-on** is an optional component that en...,"# Add-ons\n\n**Add-ons** (also called ""extensi...",# Add-ons\n\nAdd-ons are **supplementary featu...,...,* **Plug-in (computing):** Software that enhan...,"# Add-ons\n\nAdd-ons are **optional features, ...",An **add-on** is an extra or additional compon...,"""Add-ons"" (also known as ""extensions,"" ""plug-i...",**Add-ons** are supplementary software compone...,Plug-in: A software component that adds functi...,# Add-ons\n\nAdd-ons are optional extras you c...,An **add-on** is an optional extra that plugs ...,"Add-ons (also spelled add-ons, addons, extensi...",Add-ons are optional modules that extend a hos...
3,how does a dredge work?,open_qa,A dredge is essentially a floating machine des...,"A dredge is a machine used to remove sediment,...",# How a Dredge Works\n\nA dredge is a vessel o...,Dredging is an excavation activity or operatio...,A **dredge** is a machine or system designed t...,A dredge is essentially a floating machine des...,"# Dredging: Equipment, Methods, and Operations...",# How Dredging Works\n\nDredging is the proces...,...,Dredging is an underwater excavation process t...,# How Dredges Operate\n\nA dredge is a special...,A dredge is a specialized floating vessel desi...,Dredges are specialized machines designed to r...,A **dredge** is a specialized machine or syste...,Dredging is the underwater or partially underw...,# How Dredges Work\n\nA dredge is any floating...,A dredge is a floating factory built to r

In [ ]:
len(df)

5306

In [ ]:
# making pairs of claude rewrites
df_gemini_to_claude = df[['gemini_answers', 'gemini_to_claude']]
df_gemini_to_claude['source'] = 'gemini'
df_gemini_to_claude['label'] = 'claude'
df_gemini_to_claude = df_gemini_to_claude.rename(columns={'gemini_answers': 'original', 'gemini_to_claude' : 'rewrite'})

df_gpt_to_claude = df[['gpt_answers', 'gpt_to_claude']]
df_gpt_to_claude['source'] = 'gpt'
df_gpt_to_claude['label'] = 'claude'
df_gpt_to_claude = df_gpt_to_claude.rename(columns={'gpt_answers': 'original', 'gpt_to_claude' : 'rewrite'})

df_claude_to_claude = df[['claude_answers', 'claude_to_claude']]
df_claude_to_claude['source'] = 'claude'
df_claude_to_claude['label'] = 'claude'
df_claude_to_claude = df_claude_to_claude.rename(columns={'claude_answers': 'original', 'claude_to_claude' : 'rewrite'})

df_kimi_to_claude = df[['kimi_answers', 'kimi_to_claude']]
df_kimi_to_claude['source'] = 'kimi'
df_kimi_to_claude['label'] = 'claude'
df_kimi_to_claude = df_kimi_to_claude.rename(columns={'kimi_answers': 'original', 'kimi_to_claude' : 'rewrite'})

df_human_to_claude = df[['human_answers', 'human_to_claude']]
df_human_to_claude['source'] = 'human'
df_human_to_claude['label'] = 'claude'
df_human_to_claude = df_human_to_claude.rename(columns={'human_answers': 'original', 'human_to_claude' : 'rewrite'})

/tmp/ipykernel_19925/2007506323.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gemini_to_claude['source'] = 'gemini'
/tmp/ipykernel_19925/2007506323.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gemini_to_claude['label'] = 'claude'
/tmp/ipykernel_19925/2007506323.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

In [ ]:
# making pairs of gemini rewrites
df_gemini_to_gemini = df[['gemini_answers', 'gemini_to_gemini']]
df_gemini_to_gemini['source'] = 'gemini'
df_gemini_to_gemini['label'] = 'gemini'
df_gemini_to_gemini = df_gemini_to_gemini.rename(columns={'gemini_answers': 'original', 'gemini_to_gemini' : 'rewrite'})

df_gpt_to_gemini = df[['gpt_answers', 'gpt_to_gemini']]
df_gpt_to_gemini['source'] = 'gpt'
df_gpt_to_gemini['label'] = 'gemini'
df_gpt_to_gemini = df_gpt_to_gemini.rename(columns={'gpt_answers': 'original', 'gpt_to_gemini' : 'rewrite'})

df_claude_to_gemini = df[['claude_answers', 'claude_to_gemini']]
df_claude_to_gemini['source'] = 'claude'
df_claude_to_gemini['label'] = 'gemini'
df_claude_to_gemini = df_claude_to_gemini.rename(columns={'claude_answers': 'original', 'claude_to_gemini' : 'rewrite'})

df_kimi_to_gemini = df[['kimi_answers', 'kimi_to_gemini']]
df_kimi_to_gemini['source'] = 'kimi'
df_kimi_to_gemini['label'] = 'gemini'
df_kimi_to_gemini = df_kimi_to_gemini.rename(columns={'kimi_answers': 'original', 'kimi_to_gemini' : 'rewrite'})


df_human_to_gemini = df[['human_answers', 'human_to_gemini']]
df_human_to_gemini['source'] = 'human'
df_human_to_gemini['label'] = 'gemini'
df_human_to_gemini = df_human_to_gemini.rename(columns={'human_answers': 'original', 'human_to_gemini' : 'rewrite'})

/tmp/ipykernel_19925/2361799008.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gemini_to_gemini['source'] = 'gemini'
/tmp/ipykernel_19925/2361799008.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gemini_to_gemini['label'] = 'gemini'
/tmp/ipykernel_19925/2361799008.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

In [ ]:
# making pairs of gpt rewrites
df_gemini_to_gpt = df[['gemini_answers', 'gemini_to_gpt']]
df_gemini_to_gpt['source'] = 'gemini'
df_gemini_to_gpt['label'] = 'gpt'
df_gemini_to_gpt = df_gemini_to_gpt.rename(columns={'gemini_answers': 'original', 'gemini_to_gpt' : 'rewrite'})

df_gpt_to_gpt = df[['gpt_answers', 'gpt_to_gpt']]
df_gpt_to_gpt['source'] = 'gpt'
df_gpt_to_gpt['label'] = 'gpt'
df_gpt_to_gpt = df_gpt_to_gpt.rename(columns={'gpt_answers': 'original', 'gpt_to_gpt' : 'rewrite'})

df_claude_to_gpt = df[['claude_answers', 'claude_to_gpt']]
df_claude_to_gpt['source'] = 'claude'
df_claude_to_gpt['label'] = 'gpt'
df_claude_to_gpt = df_claude_to_gpt.rename(columns={'claude_answers': 'original', 'claude_to_gpt' : 'rewrite'})

df_kimi_to_gpt = df[['kimi_answers', 'kimi_to_gpt']]
df_kimi_to_gpt['source'] = 'kimi'
df_kimi_to_gpt['label'] = 'gpt'
df_kimi_to_gpt = df_kimi_to_gpt.rename(columns={'kimi_answers': 'original', 'kimi_to_gpt' : 'rewrite'})

df_human_to_gpt = df[['human_answers', 'human_to_gpt']]
df_human_to_gpt['source'] = 'human'
df_human_to_gpt['label'] = 'gpt'
df_human_to_gpt = df_human_to_gpt.rename(columns={'human_answers': 'original', 'human_to_gpt' : 'rewrite'})

/tmp/ipykernel_19925/2551824844.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gemini_to_gpt['source'] = 'gemini'
/tmp/ipykernel_19925/2551824844.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gemini_to_gpt['label'] = 'gpt'
/tmp/ipykernel_19925/2551824844.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/us

In [ ]:
# making pairs of gpt rewrites
df_gemini_to_kimi = df[['gemini_answers', 'gemini_to_kimi']]
df_gemini_to_kimi['source'] = 'gemini'
df_gemini_to_kimi['label'] = 'kimi'
df_gemini_to_kimi = df_gemini_to_kimi.rename(columns={'gemini_answers': 'original', 'gemini_to_kimi' : 'rewrite'})

df_gpt_to_kimi = df[['gpt_answers', 'gpt_to_kimi']]
df_gpt_to_kimi['source'] = 'gpt'
df_gpt_to_kimi['label'] = 'kimi'
df_gpt_to_kimi = df_gpt_to_kimi.rename(columns={'gpt_answers': 'original', 'gpt_to_kimi' : 'rewrite'})

df_claude_to_kimi = df[['claude_answers', 'claude_to_kimi']]
df_claude_to_kimi['source'] = 'claude'
df_claude_to_kimi['label'] = 'kimi'
df_claude_to_kimi = df_claude_to_kimi.rename(columns={'claude_answers': 'original', 'claude_to_kimi' : 'rewrite'})

df_kimi_to_kimi = df[['kimi_answers', 'kimi_to_kimi']]
df_kimi_to_kimi['source'] = 'kimi'
df_kimi_to_kimi['label'] = 'kimi'
df_kimi_to_kimi = df_kimi_to_kimi.rename(columns={'kimi_answers': 'original', 'kimi_to_kimi' : 'rewrite'})

df_human_to_kimi = df[['human_answers', 'human_to_kimi']]
df_human_to_kimi['source'] = 'human'
df_human_to_kimi['label'] = 'kimi'
df_human_to_kimi = df_human_to_kimi.rename(columns={'human_answers': 'original', 'human_to_kimi' : 'rewrite'})

/tmp/ipykernel_19925/3164848656.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gemini_to_kimi['source'] = 'gemini'
/tmp/ipykernel_19925/3164848656.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gemini_to_kimi['label'] = 'kimi'
/tmp/ipykernel_19925/3164848656.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable

In [ ]:
df_claude = pd.concat([df_gemini_to_claude, df_gpt_to_claude, df_claude_to_claude, df_kimi_to_claude, df_human_to_claude], axis=0, ignore_index=True)
print(len(df_claude))
print(df_claude.isna().sum())
display(df_claude.head())

26530
original    0
rewrite     6
source      0
label       0
dtype: int64


,original,rewrite,source,label
0,The most prominent composer strongly associate...,The most prominent composer associated with th...,gemini,claude
1,The Persian Wars primarily took place in **mai...,The Persian Wars primarily took place across *...,gemini,claude
2,An **add-on** is something extra or additional...,An **add-on** is an optional component that en...,gemini,claude
3,A dredge is essentially a floating machine des...,A dredge is essentially a floating machine des...,gemini,claude
4,The humanities are academic disciplines that s...,The humanities are academic disciplines that s...,gemini,claude


In [ ]:
df_gemini = pd.concat([df_gemini_to_gemini, df_gpt_to_gemini, df_claude_to_gemini, df_kimi_to_gemini, df_human_to_gemini], axis=0, ignore_index=True)
print(len(df_gemini))
print(df_gemini.isna().sum())
display(df_gemini.head())

26530
original    0
rewrite     0
source      0
label       0
dtype: int64


,original,rewrite,source,label
0,The most prominent composer strongly associate...,**György Ligeti** is the most prominent compos...,gemini,gemini
1,The Persian Wars primarily took place in **mai...,The Persian Wars primarily unfolded across mai...,gemini,gemini
2,An **add-on** is something extra or additional...,An **add-on** is an extra or additional compon...,gemini,gemini
3,A dredge is essentially a floating machine des...,A dredge is a specialized floating vessel desi...,gemini,gemini
4,The humanities are academic disciplines that s...,The humanities are academic disciplines that d...,gemini,gemini


In [ ]:
df_gpt = pd.concat([df_gemini_to_gpt, df_gpt_to_gpt, df_claude_to_gpt, df_kimi_to_gpt, df_human_to_gpt], axis=0, ignore_index=True)
print(len(df_gpt))
print(df_gpt.isna().sum())
display(df_gpt.head())

26530
original      0
rewrite     347
source        0
label         0
dtype: int64


,original,rewrite,source,label
0,The most prominent composer strongly associate...,The composer most strongly associated with the...,gemini,gpt
1,The Persian Wars primarily took place in **mai...,The Persian Wars were fought mainly in mainlan...,gemini,gpt
2,An **add-on** is something extra or additional...,An add-on is an optional extra—something attac...,gemini,gpt
3,A dredge is essentially a floating machine des...,None,gemini,gpt
4,The humanities are academic disciplines that s...,The humanities are academic disciplines that e...,gemini,gpt


In [ ]:
df_kimi = pd.concat([df_gemini_to_kimi, df_gpt_to_kimi, df_claude_to_kimi, df_kimi_to_kimi, df_human_to_kimi], axis=0, ignore_index=True).rename(columns={'gemini_answers':'original', 'gemini_to_kimi':'rewrite'})[['original', 'rewrite', 'source', 'label']]
print(len(df_kimi))
print(df_kimi.isna().sum())
display(df_kimi.head())

26530
original    0
rewrite     0
source      0
label       0
dtype: int64


,original,rewrite,source,label
0,The most prominent composer strongly associate...,György Ligeti stands as the composer most famo...,gemini,kimi
1,The Persian Wars primarily took place in **mai...,The Persian Wars unfolded across three intertw...,gemini,kimi
2,An **add-on** is something extra or additional...,An **add-on** is an optional extra that plugs ...,gemini,kimi
3,A dredge is essentially a floating machine des...,A dredge is a floating factory built to remove...,gemini,kimi
4,The humanities are academic disciplines that s...,The humanities are disciplines that investigat...,gemini,kimi


In [ ]:
print(len(df_claude))
print(len(df_kimi))
print(len(df_gemini))
print(len(df_gpt))

26530
26530
26530
26530


In [ ]:
df = pd.concat([df_claude, df_kimi, df_gemini, df_gpt], axis=0, ignore_index=True)
df.head()

,original,rewrite,source,label
0,The most prominent composer strongly associate...,The most prominent composer associated with th...,gemini,claude
1,The Persian Wars primarily took place in **mai...,The Persian Wars primarily took place across *...,gemini,claude
2,An **add-on** is something extra or additional...,An **add-on** is an optional component that en...,gemini,claude
3,A dredge is essentially a floating machine des...,A dredge is essentially a floating machine des...,gemini,claude
4,The humanities are academic disciplines that s...,The humanities are academic disciplines that s...,gemini,claude


In [ ]:
print(df.columns.tolist())

['original', 'rewrite', 'source', 'label']


In [ ]:
print(df.isna().sum())

original      0
rewrite     353
source        0
label         0
dtype: int64


In [ ]:
df_sample = (df.groupby('label', group_keys=False).apply(lambda x: x.sample(frac=0.1)).reset_index(drop=True))
len(df_sample)

/tmp/ipykernel_19925/4236159310.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = (df.groupby('label', group_keys=False).apply(lambda x: x.sample(frac=0.1)).reset_index(drop=True))


10612

In [ ]:
display(df_sample.head())
len(df_sample)

,original,rewrite,source,label
0,Airbags typically deploy **only when the crash...,Airbags deploy **only when a crash is severe e...,kimi,claude
1,Paying out a dividend can be a highly rational...,# Why Dividend Payments Are Strategically Rati...,gemini,claude
2,Belize is not “in” any other country; it **is*...,Belize is an independent sovereign nation loca...,kimi,claude
3,# Stepwise Linear Regression\n\nStepwise linea...,# Stepwise Linear Regression\n\nStepwise linea...,claude,claude
4,# VAT Tax (Value Added Tax)\n\nVAT is a **cons...,# Value Added Tax (VAT)\n\nVAT is a **consumpt...,claude,claude


10612

In [ ]:
train_df, test_df = train_test_split(df_sample, test_size=0.2, stratify=df_sample['label'])

In [ ]:
test_df.duplicated(subset=['original']).sum()

np.int64(61)

In [ ]:
df.head()
len(df)
df.to_parquet('final_long_w_human.parquet')